# **알고리즘 분석을 위한 기초**



---



## **1. 알고리즘 성능 분석 방법**
- **연산량**： 알고리즘이 얼마나 적은 연산을 수행하는가?
- **메모리 사용량**： 얼마나 적은 메모리 공간을 사용하는가?

### **@연산량 확인 방법**

|연산량 확인 방법|설명|주요 도구|
|----|----|----|
|연산 횟수 직접 카운팅 |특정 코드에서 연산 횟수를 직접 세어 연산량을 측정 | 변수 사용 (count += 1)|
|Big-O 분석을 위한 타이머 활용 | 실행 시간을 여러 입력 크기에 대해 측정하여 시간 복잡도 분석 | time 모듈 (time.time()) |
|프로파일링 도구 활용 | 함수 실행 횟수, 실행 시간 분석 | cProfile, line-profiler |
|CPU 사용량 확인 |연산량이 CPU 사용률에 미치는 영향 분석 | psutil.cpu_percent() |
|GPU(PyTorch CUDA) 메모리 | GPU VRAM, AI 모델 학습/추론 | torch.cuda.memory_allocated() |
|GPU 전체 사용량 | GPU 전체 사용량, 서버 운영, 모델 배포 | nvidia-smi |




#### **1) 연산 횟수 직접 카운팅**
알고리즘 내에서 수행되는 주요 연산(반복문, 재귀 호출, 비교 연산 등)의 횟수 센다.</br>**(전역 변수 or 데코레이터 활용)**

In [ ]:
# 버블 정렬의 연산 횟수 측정
import random

count = 0  # 연산 횟수 카운트

def bubble_sort(arr):
    global count
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            count += 1  # 비교 연산 횟수 증가
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]

arr = [random.randint(0, 100) for _ in range(10)]
print(arr)
bubble_sort(arr)
print(f"Total Operations: {count}")

#### **2) Big-O 분석을 위한 타이머 활용**
정렬 알고리즘의 실행 시간을 여러 크기의 입력에서 측정하여 Big-O 복잡도(O(n²), O(n log n) 등)를 분석할 수 있다.

In [ ]:
# 정렬 알고리즘의 시간 복잡도 비교
import time
import random

def measure_time(sort_function, size):
    arr = [random.randint(0, 10000) for _ in range(size)]
    start = time.time()  # 시간 측정 시작
    sort_function(arr)
    end = time.time()    # 시간 측정 종료
    return end - start

sizes = [100, 500, 1000, 5000]
for size in sizes:
    print(f"Size {size}: Bubble Sort = {measure_time(bubble_sort, size):.6f} sec")


- 그래프로 표현

In [ ]:
import time
import random
import matplotlib.pyplot as plt

def measure_time(sort_function, sizes):
    times = []
    for size in sizes:
        arr = [random.randint(0, 10000) for _ in range(size)]
        start = time.time()
        sort_function(arr)
        end = time.time()
        times.append(end - start)
    return times

sizes = [100, 500, 1000, 2000, 3000, 4000, 5000]
bubble_times = measure_time(bubble_sort, sizes)
plt.plot(sizes, bubble_times, label="Bubble Sort", marker='o')
plt.xlabel("Input Size")
plt.ylabel("Execution Time (seconds)")
plt.legend()
plt.title("Algorithm Complexity Analysis")
plt.show()


#### **3) 프로파일링 도구 활용**
cProfile: 각 함수가 몇 번 호출되었는지와 실행 시간을 출력해 준다.

In [ ]:
# cProfile을 이용한 함수 호출 수 측정
import cProfile

def test_function():
    total = 0
    for i in range(1000000):
        total += i
    return total

cProfile.run('test_function()')


- line_profiler: 코드 라인별 연산량 확인

In [ ]:
!pip install line-profiler

In [ ]:
from line_profiler import LineProfiler

def test_function():
    total = 0
    for i in range(1000000):  # 연산량이 많은 부분
        total += i
    return total

lp = LineProfiler()
lp.add_function(test_function)

lp.enable()   # LineProfiler: 시작
test_function()
lp.disable()  # LineProfiler: 종료

lp.print_stats()


#### **4) CPU 사용량 확인**

In [ ]:
import psutil
import time

def test_function():
    total = 0
    for i in range(1000000):
        total += i
    return total

# CPU 및 메모리 사용량 측정
process = psutil.Process()
start_cpu = process.cpu_percent(interval=None)

start_time = time.time()
test_function()
end_time = time.time()

end_cpu = process.cpu_percent(interval=None)

print(f"Execution Time: {end_time - start_time:.6f} seconds")
print(f"CPU Usage: {end_cpu:.2f}%")


#### **5) GPU 사용량 확인**
Colab에서 런타임 유형 변경 --> 예: T4.GPU 선택

In [ ]:
import tracemalloc

# ───────────────────────────────────────────
# CPU(numpy) 메모리 — tracemalloc으로 측정 가능
# ───────────────────────────────────────────
import numpy as np

tracemalloc.start()
cpu_tensor = np.random.rand(1000, 1000)   # CPU RAM에 할당
_, peak_cpu = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"CPU numpy 배열 → tracemalloc peak: {peak_cpu/1024/1024:.2f} MB ✅")

# ───────────────────────────────────────────
# GPU(PyTorch CUDA) 메모리 — 별도 API 필요
# ───────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        tracemalloc.start()
        gpu_tensor = torch.rand(1000, 1000).cuda()   # VRAM에 할당
        _, peak_gpu_trace = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        # PyTorch 전용 GPU 메모리 API
        gpu_mem = torch.cuda.memory_allocated() / 1024 / 1024

        print(f"GPU tensor → tracemalloc peak : {peak_gpu_trace/1024:.2f} KB ❌ (못 잡음)")
        print(f"GPU tensor → torch.cuda API   : {gpu_mem:.2f} MB ✅ (정확)")
    else:
        print("GPU 없음 — torch.cuda.memory_allocated() 로 VRAM 측정 필요")
except ImportError:
    print("PyTorch 미설치 — GPU 메모리는 nvidia-smi 또는 torch.cuda API로 측정")

### **[실습]: 정렬 알고리즘 시간 성능 분석**
- bubble_sort, merge_sort, builtin_sort  사용
- sizes =  [100, 500, 1000, 5000, 10000] 임의의 정수 사용
- 각 알고리즘의 실행 시간(time) 측정하여 하나의 그래프에 시각화하기

In [ ]:
# 정렬 알고리즘들 (문자열 정렬을 위해 수정 필요)
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]

def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result

def builtin_sort(arr):
    return sorted(arr)


In [ ]:
# 코드 작성
import time
import random
import matplotlib.pyplot as plt

# 시간 측정 함수
def measure_time(sort_funcs, arr):
    times = []
    for sort_func in sort_funcs:
        arr_copy = arr[:]  # 원본 배열 보호 (정렬 후 배열이 바뀌므로 복사)
        start = time.time()
        sort_func(arr_copy)
        end = time.time()
        times.append(end - start)
    return times


# 벤치마크 함수
def benchmark():
    sizes = [100, 500, 1000, 4000, 5000, 10000]
    sort_funcs = [bubble_sort, merge_sort, builtin_sort]
    func_times = [[] for _ in sort_funcs]  # 동적 리스트 생성

    # 실행 시간 측정
    for size in sizes:
        arr = [random.randint(0, 10000) for _ in range(size)]
        results = measure_time(sort_funcs, arr)
        for idx, val in enumerate(results):
            func_times[idx].append(val)

    # 그래프 그리기
    plt.figure(figsize=(8, 6))
    for idx, sort_func in enumerate(sort_funcs):
        plt.plot(sizes, func_times[idx], label=f"{sort_func.__name__}", marker='o')

    plt.xlabel("Input Size")
    plt.ylabel("Execution Time (seconds)")
    plt.title("Algorithm Time Complexity Analysis")
    plt.legend()
    plt.grid(True)
    plt.show()


# 함수 호출
benchmark()




---



### **@메모리 사용량 확인 방법**


 |메모리 사용량 확인 방법 | 설명 | 주요 도구 |
 |--------- |---------- |----- |
|(전체 프로세스)메모리 사용량 측정 | 현재 실행 중인 프로세스가 사용하는 전체 메모리 크기 확인  | psutil.memory_info()  |
|(코드 라인별)메모리 사용량 측정 | 특정 코드 라인에서 사용된 메모리 크기를 분석하여 가장 많은 메모리를 차지하는 부분 확인 |memory_profiler |
|메모리 사용량 추적 | 프로그램 실행 중 메모리 변화량을 추적하여 메모리 증가 여부 분석 |tracemalloc |
|Garbage Collector 메모리 수거 확인|Python이 자동으로 해제하지 않은 메모리를 강제로 수거하여 메모리 최적화 확인 |  gc (Garbage Collector)|



#### **1.(전체 프로세스)메모리 사용량 측정**

In [ ]:
import psutil
import time

def memory_intensive_function():
    data = [i for i in range(10**7)]  # 큰 리스트 생성
    return sum(data)

# 현재 프로세스의 메모리 사용량 측정
# - rss:Resident Set Size, 현재 프로세스가 실제로 물리적 RAM에서 차지하고 있는 메모리 크기
process = psutil.Process()
start_memory = process.memory_info().rss / (1024 * 1024)  # MB 단위

start_time = time.time()
memory_intensive_function()
end_time = time.time()

end_memory = process.memory_info().rss / (1024 * 1024)  # MB 단위

print(f"Memory Usage Before: {start_memory:.2f} MB")
print(f"Memory Usage After: {end_memory:.2f} MB")
print(f"Memory Increased: {end_memory - start_memory:.2f} MB")
print(f"Execution Time: {end_time - start_time:.6f} sec")


#### **2.(코드 라인별)메모리 사용량 측정**

In [ ]:
!pip install memory_profiler

In [ ]:
from memory_profiler import memory_usage
import time

def memory_intensive_function():
    data = [i for i in range(10**6)]  # 큰 리스트 생성
    total = sum(data)
    return total

# 메모리 사용량 측정
mem_usage, _ = memory_usage((memory_intensive_function, ), interval=0.1, retval=True)
print(f"Peak Memory Usage: {max(mem_usage):.2f} MB")


In [ ]:
'''
이 코드는 코랩에서는 IPython 환경과 호환되지 않아서 오류가 발생함
PC 파이썬 인터프리터에서 실행
- 각 코드 라인이 얼마나 많은 메모리를 사용하는지 확인 가능
'''
from memory_profiler import profile

@profile
def memory_intensive_function():
    data = [i for i in range(10**6)]  # 큰 리스트 생성
    total = sum(data)
    return total

memory_intensive_function()


#### **3.메모리 사용량 추적**

In [ ]:
import tracemalloc

# ① Python 객체 — 측정됨
tracemalloc.start()
my_list = [i for i in range(50000)]   # 힙에 할당
_, peak1 = tracemalloc.get_traced_memory()
tracemalloc.stop()

# ② 단순 루프 변수 — 거의 측정 안 됨
tracemalloc.start()
total = 0
for i in range(50000):               # 스택 변수, 새 객체 없음
    total += i
_, peak2 = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"리스트 생성 (힙 할당)  → peak: {peak1/1024:.2f} KB")
print(f"단순 루프 합산 (스택)  → peak: {peak2/1024:.2f} KB")


 #### **4. 메모리 누수 위치 추적 (traceback 활용)**

In [ ]:
import tracemalloc

tracemalloc.start()

# 어디서 메모리를 많이 쓰는지 모를 때
def make_big_data():
    return {str(i): [j**2 for j in range(100)] for i in range(500)}

result = make_big_data()

# 상위 3개 메모리 할당 지점 출력
snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')

print("=== 메모리 많이 쓰는 코드 위치 Top 3 ===")
for stat in top_stats[:3]:
    print(f"{stat.traceback.format()[0]}")
    print(f"  → 할당량: {stat.size / 1024:.2f} KB\n")

tracemalloc.stop()

#### **5.Garbage Collector 메모리 수거 확인**

In [ ]:
import gc
import psutil

def create_large_data():
    return [i for i in range(10**6)]  # 큰 리스트 생성

process = psutil.Process()
print(f"Memory Before: {process.memory_info().rss / (1024 * 1024):.2f} MB")

data = create_large_data()

print(f"Memory After: {process.memory_info().rss / (1024 * 1024):.2f} MB")

del data  # 메모리 해제
gc.collect()  # 가비지 컬렉션 실행

print(f"Memory After GC: {process.memory_info().rss / (1024 * 1024):.2f} MB")




---



### **[실습]: 정렬 알고리즘 성능 분석 (시간, 메모리)**
- bubble_sort, merge_sort, builtin_sort  사용
- sizes =  [100, 500, 1000, 5000, 10000] 임의의 정수 사용
- 실행 시간(time), **전체 메모리 사용량 측정하여  그래프에 시각화하기**

In [ ]:
import time
import random
import matplotlib.pyplot as plt
import gc
import tracemalloc  # 파이썬 내부 메모리 할당 추적 라이브러리

# [정렬 알고리즘 생략 - 기존과 동일]

def measure_performance(sort_funcs, arr):
    times = []
    memory_peaks = []

    for sort_func in sort_funcs:
        gc.collect() # 측정 전 가비지 컬렉션
        arr_copy = list(arr) # 원본 보호

        # [핵심] tracemalloc 시작
        tracemalloc.start()
        start_time = time.time()

        # 알고리즘 실행
        sort_func(arr_copy)

        end_time = time.time()
        # [핵심] 현재 메모리 사용량과 피크(최대) 사용량을 가져옴
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        times.append(end_time - start_time)
        memory_peaks.append(peak / (1024 * 1024))  # MB 단위 변환

        del arr_copy
        gc.collect()

    return times, memory_peaks

def benchmark():
    # 버블 정렬의 O(n^2) 특성상 n이 너무 크면 시간이 오래 걸리므로 적절히 조절
    sizes = [100, 500, 1000, 3000, 5000]
    sort_funcs = [bubble_sort, merge_sort, builtin_sort]
    func_times = [[] for _ in sort_funcs]
    func_memorys = [[] for _ in sort_funcs]

    for size in sizes:
        arr = [random.randint(0, 10000) for _ in range(size)]
        t_results, m_results = measure_performance(sort_funcs, arr)
        for idx in range(len(sort_funcs)):
            func_times[idx].append(t_results[idx])
            func_memorys[idx].append(m_results[idx])

    # 시각화 (1행 2열)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax1, ax2 = axes

    markers = ['o', 's', '^']
    for idx, sort_func in enumerate(sort_funcs):
        ax1.plot(sizes, func_times[idx], label=f"{sort_func.__name__}", marker=markers[idx])
        ax2.plot(sizes, func_memorys[idx], label=f"{sort_func.__name__}", marker=markers[idx])

    ax1.set_title("Time Complexity (seconds)")
    ax1.legend(); ax1.grid(True)

    ax2.set_title("Peak Memory Usage (MB)")
    ax2.set_ylabel("MB")
    ax2.legend(); ax2.grid(True)

    plt.tight_layout()
    plt.savefig("sorting_performance.png")
    plt.show()


if __name__ == "__main__":
    benchmark()

- **문자열 데이터 사용**

In [ ]:
import random

# 실제 의미있는 데이터 풀 생성
class RealDataGenerator:
    def __init__(self):

        # IT 회사 명단
        self.companies = [
            "삼성전자", "LG전자", "SK하이닉스", "네이버", "카카오", "라인", "쿠팡",
            "배달의민족", "토스", "당근마켓", "야놀자", "마켓컬리", "직방",
            "우아한형제들", "NHN", "엔씨소프트", "넷마블", "스마일게이트",
            "크래프톤", "컴투스", "위메이드", "펄어비스"
        ]

        # 프로그래밍 언어 및 기술
        self.technologies = [
            "Python", "Java", "JavaScript", "TypeScript", "C++", "C#", "Go", "Rust",
            "React", "Vue.js", "Angular", "Node.js", "Spring", "Django", "Flask",
            "Docker", "Kubernetes", "AWS", "Azure", "GCP", "MongoDB", "MySQL",
            "PostgreSQL", "Redis", "Elasticsearch", "Kafka", "RabbitMQ", "Jenkins"
        ]

        # 도시 명단
        self.cities = [
            "서울", "부산", "대구", "인천", "광주", "대전", "울산", "수원", "용인", "고양",
            "창원", "성남", "청주", "안산", "전주", "천안", "남양주", "화성", "평택", "의정부",
            "시흥", "파주", "김해", "순천", "목포", "포항", "구미", "경주", "양산", "진주"
        ]

        # 직무 분야
        self.job_fields = [
            "프론트엔드개발자", "백엔드개발자", "풀스택개발자", "데이터사이언티스트", "AI엔지니어",
            "DevOps엔지니어", "클라우드엔지니어", "보안전문가", "모바일개발자", "게임개발자",
            "데이터엔지니어", "머신러닝엔지니어", "블록체인개발자", "QA엔지니어", "시스템관리자"
        ]


    def generate_company_data(self, size):
        """회사 관련 데이터 생성"""
        departments = ["개발팀", "AI팀", "데이터팀", "인프라팀", "보안팀", "기획팀"]
        return [
            f"{random.choice(self.companies)}_{random.choice(departments)}_{random.choice(self.technologies)}개발자_{i:05d}"
            for i in range(size)
        ]

    def generate_project_data(self, size):
        """프로젝트 관련 데이터 생성"""
        project_types = ["웹서비스", "모바일앱", "AI시스템", "데이터분석", "클라우드인프라", "게임"]
        return [
            f"{random.choice(project_types)}_{random.choice(self.cities)}지역_{random.choice(self.technologies)}기반_프로젝트{i:06d}"
            for i in range(size)
        ]

    def generate_job_posting_data(self, size):
        """채용공고 관련 데이터 생성"""
        experience_levels = ["신입", "1-3년", "3-5년", "5년이상", "시니어"]
        return [
            f"{random.choice(self.companies)}_{random.choice(self.job_fields)}_{random.choice(experience_levels)}경력_채용공고{i:05d}"
            for i in range(size)
        ]

    def generate_course_data(self, size):
        """강의 관련 데이터 생성"""
        course_types = ["기초", "중급", "고급", "실무", "프로젝트"]
        return [
            f"{random.choice(self.technologies)}_{random.choice(course_types)}과정_{random.choice(self.universities)}_{2024}년{random.randint(1,12):02d}월_수강생{i:04d}"
            for i in range(size)
        ]

In [ ]:
import time
import matplotlib.pyplot as plt
import gc
import tracemalloc  # 파이썬 내부 메모리 할당 추적 라이브러리

# --- 성능 측정 함수 (Peak Memory 포착 버전) ---
def measure_performance(sort_funcs, arr):
    times = []
    memory_peaks = []

    for sort_func in sort_funcs:
        gc.collect()
        arr_copy = list(arr)

        tracemalloc.start()  # 메모리 추적 시작
        start_time = time.time()

        sort_func(arr_copy)

        end_time = time.time()
        _, peak = tracemalloc.get_traced_memory() # 피크 메모리 획득
        tracemalloc.stop()   # 메모리 추적 종료

        times.append(end_time - start_time)
        memory_peaks.append(peak / (1024 * 1024)) # MB 단위

        del arr_copy
        gc.collect()

    return times, memory_peaks

# --- 벤치마크 및 시각화 ---
def benchmark():
    # 문자열 정렬은 숫자보다 연산 비용이 크므로 사이즈를 적절히 조절 (MIT 학생들에겐 O(n^2)의 한계를 보여주기 좋음)
    sizes = [100, 500, 1000, 2000, 3000]
    sort_funcs = [bubble_sort, merge_sort, builtin_sort]
    func_times = [[] for _ in sort_funcs]
    func_memorys = [[] for _ in sort_funcs]

    data_gen = RealDataGenerator()

    for size in sizes:
        # 실무 데이터 생성 (IT 회사 직원 데이터 예시)
        arr = data_gen.generate_company_data(size)

        time_results, memory_results = measure_performance(sort_funcs, arr)
        for idx in range(len(sort_funcs)):
            func_times[idx].append(time_results[idx])
            func_memorys[idx].append(memory_results[idx])

    # 그래프 그리기
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    for idx, sort_func in enumerate(sort_funcs):
        axes[0].plot(sizes, func_times[idx], label=f"{sort_func.__name__}", marker='o')
        axes[1].plot(sizes, func_memorys[idx], label=f"{sort_func.__name__}", marker='s')

    axes[0].set_title("Time Complexity: Real-world String Sorting")
    axes[0].set_ylabel("Execution Time (sec)")
    axes[1].set_title("Space Complexity: Peak Memory Usage")
    axes[1].set_ylabel("Memory Usage (MB)")

    for ax in axes:
        ax.set_xlabel("Input Size (Number of Employee Records)")
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    benchmark()



---



## ====================

## **2. 피보나치 구현 방식별 성능 측정**

In [ ]:
"""
==============================================================
 피보나치 구현 방식별 성능 측정
 - 측정 항목: 연산 횟수, 실행 시간(ms), 힙 메모리 (현재 / 최대)
 - 측정 도구: tracemalloc (Python 힙), time.perf_counter
==============================================================
"""

import time
import tracemalloc
import sys
import itertools
from functools import lru_cache

sys.setrecursionlimit(5000)


# ──────────────────────────────────────────────────────────────
# 연산 횟수 카운터
# ──────────────────────────────────────────────────────────────
call_counter = {}
memo_dict    = {}

def count_calls(name):
    call_counter[name] = call_counter.get(name, 0) + 1


# ==============================================================
# 피보나치 구현 4종
# ==============================================================

# 1. 순수 재귀  O(2^n) / O(n)
def recursive_fibonacci(n, _name="recursive"):
    count_calls(_name)
    if n <= 1:
        return n
    return recursive_fibonacci(n-1, _name) + recursive_fibonacci(n-2, _name)


# 2. 메모이제이션  O(n) / O(n)
@lru_cache(maxsize=None)
def _memo_fib_inner(n):
    if n <= 1:
        return n
    return _memo_fib_inner(n-1) + _memo_fib_inner(n-2)

def memoized_fibonacci(n, _name="memoized"):
    count_calls(_name)
    return _memo_fib_inner(n)


# 3. 반복문 Bottom-up  O(n) / O(1)
def iterative_fibonacci(n, _name="iterative"):
    count_calls(_name)
    if n <= 1:
        return n
    prev2, prev1 = 0, 1
    for _ in range(2, n + 1):
        count_calls(_name)
        current      = prev1 + prev2
        prev2, prev1 = prev1, current
    return prev1


# 4. 행렬 거듭제곱  O(log n) / O(log n)
def _mat_mul(A, B, _name):
    count_calls(_name)
    return [
        [A[0][0]*B[0][0] + A[0][1]*B[1][0],
         A[0][0]*B[0][1] + A[0][1]*B[1][1]],
        [A[1][0]*B[0][0] + A[1][1]*B[1][0],
         A[1][0]*B[0][1] + A[1][1]*B[1][1]],
    ]

def _mat_pow(mat, n, _name):
    count_calls(_name)
    if n == 1:
        return mat
    if n % 2 == 0:
        half = _mat_pow(mat, n // 2, _name)
        return _mat_mul(half, half, _name)
    return _mat_mul(mat, _mat_pow(mat, n - 1, _name), _name)

def matrix_fibonacci(n, _name="matrix"):
    if n <= 1:
        return n
    return _mat_pow([[1, 1], [1, 0]], n, _name)[0][1]


# ──────────────────────────────────────────────────────────────
# ▼ 여기만 수정하면 전체 테스트가 자동으로 바뀝니다 ▼
# ──────────────────────────────────────────────────────────────

# 방식 등록 (순서가 출력 순서)
FIB_FUNCS = {
    "recursive" : recursive_fibonacci,
    "memoized"  : memoized_fibonacci,
    "iterative" : iterative_fibonacci,
    "matrix"    : matrix_fibonacci,
}

# 테스트 n값 묶음
SMALL_NS = [10, 20, 30]          # 4종 모두 비교
LARGE_NS = [100, 500, 1000]      # 재귀 제외 비교

# 큰 n에서 제외할 방식 (재귀는 n=30 이상에서 수십 초 소요)
LARGE_SKIP = {"recursive"}

# 이론적 복잡도 (lambda n → 문자열)
COMPLEXITY = {
    "recursive" : ("O(2^n)",   "O(n)",      lambda n: 2**n - 1,
                   "콜스택 깊이 n"),
    "memoized"  : ("O(n)",     "O(n)",      lambda n: 1,
                   "캐시 딕셔너리 n개"),
    "iterative" : ("O(n)",     "O(1)",      lambda n: n + 1,
                   "변수 2개만 사용"),
    "matrix"    : ("O(log n)", "O(log n)",  lambda n: n.bit_length() * 2,
                   "재귀 깊이 log n"),
}

# ──────────────────────────────────────────────────────────────
# 상태 초기화 (매 측정 전 호출)
# ──────────────────────────────────────────────────────────────
def reset_state(name):
    call_counter[name] = 0
    memo_dict.clear()
    _memo_fib_inner.cache_clear()


# ==============================================================
# 측정 함수
# ==============================================================
def measure(name, func, n):
    """단일 실행 → 연산횟수 / 시간 / 메모리 측정 → dict 반환"""
    reset_state(name)

    tracemalloc.start()
    t0     = time.perf_counter()
    result = func(n)
    t1     = time.perf_counter()
    cur, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        "name"       : name,
        "n"          : n,
        "result"     : result,
        "calls"      : call_counter[name],
        "time_ms"    : round((t1 - t0) * 1000, 4),
        "cur_bytes"  : cur,
        "peak_bytes" : peak,
    }


def run_benchmark(ns, skip=None):
    """
    ns   : 테스트할 n값 리스트
    skip : 제외할 방식 이름 집합 (기본값 없음)
    → itertools.product 로 (n, 방식) 조합 자동 생성
    """
    skip = skip or set()
    return [
        measure(name, func, n)
        for n, (name, func) in itertools.product(ns, FIB_FUNCS.items())
        if name not in skip
    ]


# ==============================================================
# 출력 함수
# ==============================================================
SEP  = "─" * 82
SEP2 = "=" * 82
HDR  = (f"{'방식':<12} {'n':>5} {'F(n)':>10} {'연산횟수':>10} "
        f"{'시간(ms)':>10} {'현재메모리(B)':>14} {'최대메모리(B)':>14}")


def print_results(results, title=""):
    print(f"\n{SEP2}")
    if title:
        print(f"  {title}")
        print(SEP2)
    print(HDR)
    print(SEP)

    prev_n = None
    for r in results:
        if prev_n is not None and r["n"] != prev_n:
            print()                          # n 그룹 사이 빈 줄
        # F(n)이 너무 길면 자름
        fib_str = str(r["result"])
        fib_str = fib_str[:9] + "…" if len(fib_str) > 10 else fib_str
        print(
            f"{r['name']:<12} {r['n']:>5} {fib_str:>10} "
            f"{r['calls']:>10} {r['time_ms']:>10.4f} "
            f"{r['cur_bytes']:>14} {r['peak_bytes']:>14}"
        )
        prev_n = r["n"]
    print(SEP)


def print_complexity_table():
    print(f"\n{'구현방식':<12} {'시간':^10} {'공간':^10} {'이론 연산횟수(n=20)':>20} {'특징'}")
    print(SEP)
    n = 20
    for name, (tc, sc, formula, note) in COMPLEXITY.items():
        theory_val = formula(n)
        print(f"{name:<12} {tc:^10} {sc:^10} {theory_val:>20} {note}")
    print(SEP)


def print_theory_vs_actual(results, n=20):
    actuals = {r["name"]: r["calls"] for r in results if r["n"] == n}

    print(f"\n[이론값 vs 실측값 — n={n}]")
    print(f"{'방식':<12} {'이론값':>12} {'실측값':>12} {'오차':>8}")
    print(SEP)
    for name, (_, _, formula, _) in COMPLEXITY.items():
        if name not in actuals:
            continue
        theory  = formula(n)
        actual  = actuals[name]
        diff    = actual - theory
        print(f"{name:<12} {theory:>12} {actual:>12} {diff:>+8}")
    print(SEP)


# ==============================================================
# 메인
# ==============================================================
if __name__ == "__main__":

    # ── 소형 n: 4종 모두 ──────────────────────────────────────
    small_results = run_benchmark(SMALL_NS)
    print_results(small_results, "피보나치 4종 비교  (n = 10, 20, 30)")
    print_complexity_table()
    print_theory_vs_actual(small_results, n=20)

    # ── 대형 n: 재귀 제외 ─────────────────────────────────────
    large_results = run_benchmark(LARGE_NS, skip=LARGE_SKIP)
    print_results(large_results,
                  f"대형 n 비교  (재귀 제외: {LARGE_SKIP})  n = {LARGE_NS}")

    print("\n측정 완료!")